In [ ]:
import numpy as np
import ioh

class LLM_CMAES:
    '''
    CMA-ES variant that injects a small fraction of heavy-tailed Student-t samples
    into the Gaussian offspring each generation. Injected Student-t directions are
    rescaled so their Euclidean norm matches the expected norm of a standard
    Gaussian draw (preserving directional heavy tails while limiting extreme radii).
    Additionally implements a stagnation-triggered temporary increase in the
    heavy-tail injection fraction; the magnitude of this boost is adaptive to the
    stagnation length (longer stagnation -> larger boost up to a cap).

    New in Cycle 10: Adaptive tail heaviness. The degrees-of-freedom (nu) of the
    Student-t used for injected samples is nu_base during normal progress and,
    when a stagnation-triggered boost is active, is reduced (heavier tails) by
    1 every K stagnation generations down to nu_min. This is applied only during
    the same boost window used by the adaptive alpha.

    Cycle 15 addition: Transient, stagnation-triggered population-size increase
    (lambda boost) that temporarily samples more offspring for T generations
    when stagnation is detected. The total evaluation budget is preserved by
    shortening the remaining number of generations proportionally so that no
    extra objective evaluations occur overall.
    '''
    def __init__(self, pop_size=20, num_iterations=100, sigma=0.5):
        self.pop_size = pop_size
        self.num_iterations = num_iterations
        self.sigma = sigma

        # Base fraction of Student-t offspring
        self._alpha_base = 0.10

        # Adaptive tail-heaviness parameters
        self._nu_base = 6
        self._nu_min = 2
        self._nu_k = 2

        # Stagnation-triggered adaptive injection parameters
        self._alpha_max = 0.30
        self._alpha_k = 0.05
        self._stagnation_threshold = 5
        self._alpha_boost_duration = 3

        # Transient population-size increase parameters
        self._lambda_boost_factor = 1.8

        self.variant_description = {
            "sampling": (
                "Modified: Adaptive heavy-tail injection fraction and adaptive tail heaviness, "
                "plus a transient, stagnation-triggered population-size (lambda) increase. "
                "Normally use alpha_base=0.10 of offspring replaced by multivariate Student-t draws. "
                "If no best-so-far improvement is observed for S=5 consecutive generations, start "
                "a boost of fixed duration T=3 generations. The heavy-tail injection boost magnitude "
                "is adaptive to stagnation length: alpha_boost = min(alpha_max, alpha_base + alpha_k * (stagnation_counter - S + 1)). "
                "During the same boost window, the Student-t degrees-of-freedom nu is reduced to make tails heavier: "
                "nu = max(nu_min, nu_base - floor((stagnation_counter - S + 1)/K)). "
                "Additionally (Cycle 15), start a transient lambda increase during the boost window by a factor "
                "lambda_factor=_lambda_boost_factor (rounded up to integer offspring). The total evaluation budget "
                "(lambda_base * num_iterations) is preserved by shortening remaining generations proportionally so that "
                "no extra objective evaluations are performed. Injected Student-t vectors are rescaled so their Euclidean "
                "norm equals the expected norm of a standard Gaussian draw. Injected offspring replace existing ones "
                "within the per-generation budget; no extra evaluations are performed."
            ),
            "mean_update": "Unchanged from parent algorithm.",
            "adaptation": "Unchanged from parent algorithm."
        }

    def __call__(self, problem: ioh.problem.RealSingleObjective):
        dim = problem.meta_data.n_variables
        low_b, high_b = problem.bounds.lb[0], problem.bounds.ub[0]

        lambda_base = 4 + int(3 * np.log(dim)) if self.pop_size is None else self.pop_size
        total_budget = int(lambda_base * self.num_iterations)

        expected_norm = np.sqrt(dim) * (1 - 1 / (4 * dim) + 1 / (21 * dim ** 2))

        mean = np.random.uniform(low_b, high_b, dim)
        sigma = self.sigma * (high_b - low_b)
        C = np.eye(dim)
        p_sigma = np.zeros(dim)
        p_c = np.zeros(dim)

        best_solution = None
        best_fitness = np.inf
        history = {
            "best_fitness": [], "sigma": [], "mean": [],
            "stagnation_counter": 0, "restart_count": 0
        }

        boost_remaining = 0
        boost_alpha_current = None
        lambda_boost_current = None
        evals_done = 0
        iteration = 0

        while evals_done < total_budget and iteration < self.num_iterations:
            C = (C + C.T) / 2
            if not np.all(np.isfinite(C)):
                C = np.eye(dim)
            C += 1e-12 * np.eye(dim)

            eigenvalues, B = np.linalg.eigh(C)
            eigenvalues = np.maximum(eigenvalues, 1e-20)
            D = np.sqrt(eigenvalues)
            BD = B @ np.diag(D)

            # ========================================================
            # LLM SECTION 1: Sampling
            # ========================================================
            if history["stagnation_counter"] >= self._stagnation_threshold and boost_remaining == 0:
                extra = history["stagnation_counter"] - self._stagnation_threshold + 1
                adaptive_alpha = min(self._alpha_max, self._alpha_base + self._alpha_k * extra)
                boost_alpha_current = adaptive_alpha
                
                lambda_boost_current = int(np.ceil(lambda_base * self._lambda_boost_factor))
                lambda_boost_current = max(1, min(lambda_boost_current, total_budget - evals_done))
                boost_remaining = self._alpha_boost_duration

            current_alpha = boost_alpha_current if (boost_remaining > 0 and boost_alpha_current is not None) else self._alpha_base

            if boost_remaining > 0 and boost_alpha_current is not None and history["stagnation_counter"] >= self._stagnation_threshold:
                extra = history["stagnation_counter"] - self._stagnation_threshold + 1
                decrements = extra // self._nu_k
                current_nu = max(self._nu_min, self._nu_base - int(decrements))
            else:
                current_nu = float(self._nu_base)

            lambda_gen = lambda_boost_current if (boost_remaining > 0 and lambda_boost_current is not None) else lambda_base
            remaining_budget = total_budget - evals_done
            lambda_gen = max(1, min(lambda_gen, int(remaining_budget)))

            mu = max(1, lambda_gen // 2)
            weights = np.log(mu + 0.5) - np.log(np.arange(1, mu + 1))
            weights /= np.sum(weights)
            mu_eff = 1.0 / np.sum(weights ** 2)

            c_sigma = (mu_eff + 2) / (dim + mu_eff + 5)
            d_sigma = 1 + 2 * max(0, np.sqrt((mu_eff - 1) / (dim + 1)) - 1) + c_sigma
            c_c = (4 + mu_eff / dim) / (dim + 4 + 2 * mu_eff / dim)
            c1 = 2 / ((dim + 1.3) ** 2 + mu_eff)
            c_mu = min(1 - c1, 2 * (mu_eff - 2 + 1 / mu_eff) / ((dim + 2) ** 2 + mu_eff))

            z = np.random.randn(lambda_gen, dim)

            if current_alpha > 0.0:
                n_heavy = int(np.ceil(current_alpha * lambda_gen))
                n_heavy = max(0, min(n_heavy, lambda_gen))
                if n_heavy > 0:
                    heavy_idx = np.random.choice(lambda_gen, size=n_heavy, replace=False)
                    df = max(float(current_nu), float(self._nu_min))
                    z_t = np.random.standard_t(df, size=(n_heavy, dim))
                    norms = np.linalg.norm(z_t, axis=1)
                    scales = expected_norm / np.maximum(norms, 1e-12)
                    z[heavy_idx] = z_t * scales[:, np.newaxis]

            if boost_remaining > 0:
                boost_remaining -= 1
                if boost_remaining == 0:
                    boost_alpha_current, lambda_boost_current = None, None

            population = mean + sigma * (z @ BD.T)

            if not np.all(np.isfinite(population)):
                population = np.nan_to_num(population, nan=0.0, posinf=high_b, neginf=low_b)
            population = np.clip(population, low_b, high_b)

            # ========================================================
            # END LLM SECTION 1
            # ========================================================

            fitness = np.array([problem(ind) for ind in population])
            evals_done += lambda_gen

            sorted_indices = np.argsort(fitness)
            population = population[sorted_indices]
            z = z[sorted_indices]
            fitness = fitness[sorted_indices]

            if fitness[0] < best_fitness:
                best_fitness = fitness[0]
                best_solution = population[0].copy()
                history["stagnation_counter"] = 0
                boost_remaining, boost_alpha_current, lambda_boost_current = 0, None, None
            else:
                history["stagnation_counter"] += 1

            selected_population = population[:mu]
            old_mean = mean.copy()

            # ========================================================
            # LLM SECTION 2: Mean update
            # ========================================================
            mean = np.clip(np.sum(weights[:, np.newaxis] * selected_population, axis=0), low_b, high_b)
            # ========================================================
            # END LLM SECTION 2
            # ========================================================

            # ========================================================
            # LLM SECTION 3: Adaptation
            # ========================================================
            C_inv_sqrt = B @ np.diag(1 / D) @ B.T

            p_sigma = (1 - c_sigma) * p_sigma + np.sqrt(c_sigma * (2 - c_sigma) * mu_eff) * (C_inv_sqrt @ ((mean - old_mean) / sigma))

            sigma_exponent = np.clip((c_sigma / d_sigma) * (np.linalg.norm(p_sigma) / expected_norm - 1), -50.0, 50.0)
            sigma *= np.exp(sigma_exponent)

            if not np.isfinite(sigma):
                sigma = self.sigma * (high_b - low_b)
            sigma = np.clip(sigma, 1e-12, 1e3 * (high_b - low_b))

            h_sigma = 1.0 if (np.linalg.norm(p_sigma) / np.sqrt(1 - (1 - c_sigma) ** (2 * (iteration + 1))) < (1.4 + 2 / (dim + 1)) * expected_norm) else 0.0

            p_c = (1 - c_c) * p_c + h_sigma * np.sqrt(c_c * (2 - c_c) * mu_eff) * ((mean - old_mean) / sigma)

            y = (selected_population - old_mean) / sigma
            rank_mu_update = np.sum([weights[i] * np.outer(y[i], y[i]) for i in range(mu)], axis=0)

            C = (1 - c1 - c_mu) * C + c1 * np.outer(p_c, p_c) + c_mu * rank_mu_update
            C = (C + C.T) / 2

            if not np.all(np.isfinite(C)):
                C = np.eye(dim)
            C += 1e-12 * np.eye(dim)
            # ========================================================
            # END LLM SECTION 3
            # ========================================================

            history["best_fitness"].append(best_fitness)
            history["sigma"].append(sigma)
            history["mean"].append(mean.copy())
            iteration += 1

        return best_solution, best_fitness[cite: 1]